# Arm G cross-layer rank-4 causal mediation

Layer-27 rank expansion saturated: rank 4 and rank 8 attenuate the natural
condition contrast almost identically while removing only about 3.25% of it.
This run asks whether the missing causal mass is distributed through depth.

The same paired rank-4 construction is learned independently at six frozen
hidden-state layers (12, 16, 20, 24, 27, 30) from seed-101 and seed-102
development-family differences, then evaluated on fresh seed-106 prompts from
the two held-out families. Each layer is ablated alone, and then all six are
ablated cumulatively in ascending depth order with mapping centers recaptured
under the active upstream cascade. Layer 12 sits below the decodability onset
in both source seeds and is the frozen pre-onset specificity layer.

Primary question: does the full six-layer ablation attenuate the condition
contrast by more than the layer-27 ablation alone?

Set **Runtime → Change runtime type → A100 GPU**, then run the cells in order.
The run is roughly 880 batched forward passes over 128 evaluation prompts;
expect about 15-25 minutes on an A100. There is no generation and no
checkpointing, so a disconnect means restarting the run.

In [ ]:
# Colab supplies torch/CUDA.
print("Protocol: ARM_G_CROSS_LAYER_MEDIATION_V1")
%pip -q install "transformers==5.0.0" "accelerate==1.12.0" \
  "sentence-transformers==5.2.2" "scikit-learn==1.8.0"

In [ ]:
# Mount Drive and copy the frozen launch files to local Colab storage.
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
LAUNCH_DIR = "/content/drive/MyDrive/phi-map/arm-g-cross-layer-launch"
LAUNCH_FILES = (
    "arm_g_cross_layer.py",
    "arm_g_causal_subspace.py",
    "arm_g_causal_dose_ablation.py",
    "arm_g_causal.py",
    "arm_g_phase1.py",
    "arm_g_scenarios.py",
)
for name in LAUNCH_FILES:
    source = f"{LAUNCH_DIR}/{name}"
    assert os.path.exists(source), f"Missing {source}"
    shutil.copy2(source, f"/content/{name}")
print("Arm G cross-layer launch files staged: OK")

In [ ]:
# Frozen run configuration.
ACTING_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
WORK_DIR = "/content/drive/MyDrive/phi-map/arm-g-cross-layer-seed106-v1"
PAIRS_PER_FAMILY = 16
BOOTSTRAP = 2000
RANDOM_SUBSPACES = 8
BATCH_SIZE = 16

# Put HF_TOKEN in Colab's Secrets panel; do not paste it into the notebook.
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add an HF_TOKEN secret with Llama-3.1-8B-Instruct access"

In [ ]:
# Hardware and gated-model access gate.
from huggingface_hub import hf_hub_download
hf_hub_download(ACTING_MODEL, "config.json", token=HF_TOKEN)
print("Hugging Face model access: OK")

import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1024**3, 1), "GiB")
assert "A100" in props.name and props.total_memory >= 35 * 1024**3
assert torch.cuda.is_bf16_supported()

In [ ]:
# Deterministic protocol tests: basis construction, mapping-centered
# ablation, hook ordering in the cascade, and the frozen decision rule.
import os, subprocess, sys
env = dict(os.environ)
env["HF_TOKEN"] = HF_TOKEN
base_cmd = [
    sys.executable, "/content/arm_g_cross_layer.py",
    "--model", ACTING_MODEL,
    "--output-dir", WORK_DIR,
    "--pairs-per-family", str(PAIRS_PER_FAMILY),
    "--bootstrap", str(BOOTSTRAP),
    "--random-subspaces", str(RANDOM_SUBSPACES),
    "--batch-size", str(BATCH_SIZE),
]
subprocess.run(base_cmd + ["--self-test"], check=True, env=env)

In [ ]:
# Learn the frozen per-layer rank-4 bases, then run single-layer and
# cumulative cascade ablations with matched random-subspace controls.
subprocess.run(base_cmd, check=True, env=env)

In [ ]:
# Compact result view. Full per-row margins and projections remain in Drive.
import json
result_path = f"{WORK_DIR}/arm_g_cross_layer_result.json"
result = json.load(open(result_path))
layers = sorted(int(key) for key in result["single_layer"])
summary = {
    "decision": result["decision"],
    "decision_reasons": result["decision_reasons"],
    "baseline_condition_contrast": result["baseline"]["condition_contrast"][
        "overall"
    ],
    "single_layer": {
        layer: {
            "attenuation": item["attenuation"]["overall"],
            "attenuation_fraction": item["attenuation_fraction"],
            "ci_95": item["attenuation_bootstrap"]["ci_95"],
            "random_absolute_p95": item["random_control"]["absolute_p95"],
            "exceeds_random_control": item["exceeds_random_control"],
            "decodability": item["decodability"],
        }
        for layer, item in result["single_layer"].items()
    },
    "cumulative": {
        layer: {
            "layers_ablated": item["layers_ablated"],
            "attenuation": item["attenuation"]["overall"],
            "attenuation_fraction": item["attenuation_fraction"],
            "ci_95": item["attenuation_bootstrap"]["ci_95"],
        }
        for layer, item in result["cumulative"].items()
    },
    "depth_increment": result["depth_increment"],
    "specificity": result["specificity"],
    "sample_counts": result["sample_counts"],
}
print(json.dumps(summary, indent=2))

In [ ]:
# Archive a compact summary and a compressed full artifact for the repo.
import base64, gzip, json

summary_path = f"{WORK_DIR}/arm_g_cross_layer_result_summary.json"
archive_path = f"{WORK_DIR}/arm_g_cross_layer_result.json.gz.b64"
with open(summary_path, "w") as handle:
    json.dump(
        {
            **summary,
            "protocol": {
                "source_seeds": [101, 102],
                "evaluation_seed": 106,
                "layers": layers,
                "rank": 4,
                "random_subspaces": RANDOM_SUBSPACES,
                "bootstrap_repetitions": BOOTSTRAP,
            },
            "decodability_profile": result["decodability_profile"],
            "full_result_artifact": "arm_g_cross_layer_result.json.gz.b64",
        },
        handle,
        indent=1,
    )
raw = json.dumps(result).encode("utf-8")
with open(archive_path, "wb") as handle:
    handle.write(base64.b64encode(gzip.compress(raw)))
print("summary:", summary_path)
print("archive:", archive_path)